In [ ]:
# Colab reproducibility — uncomment to clone the repo and enter this folder:
# !git clone https://github.com/JuliettKhar/mlops-zoomcamp-exercises.git
# %cd mlops-zoomcamp-exercises/02-experiment-tracking

In [20]:
import mlflow

In [32]:
# Q1
print(mlflow.__version__)

3.12.0


In [22]:
green_taxi_01 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-01.parquet'
green_taxi_02 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-02.parquet'
green_taxi_03 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-03.parquet'

In [23]:
import os
import pickle
import click
import pandas as pd

In [24]:
from sklearn.feature_extraction import DictVectorizer

In [25]:
def dump_pickle(obj, filename: str):
    with open(filename, 'wb') as f_out:
        return pickle.dump(obj, f_out)


In [26]:
def read_data_frame(filename: str):
    data_frame = pd.read_parquet(filename)
    data_frame['duration'] = data_frame['lpep_dropoff_datetime'] - data_frame['lpep_pickup_datetime']
    data_frame.duration = data_frame.duration.apply(lambda td: td.total_seconds() / 60)
    data_frame = data_frame[(data_frame.duration >= 1)& (data_frame.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    data_frame[categorical] = data_frame[categorical].astype(str)

    return data_frame

In [27]:
def preprocess(data_frame: pd.DataFrame, dict_vector: DictVectorizer, fit_dict_vect: bool = False):
    data_frame['PU_DO'] = data_frame['PULocationID'] + '_' + data_frame['DOLocationID'] 
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    
    dicts = data_frame[categorical + numerical].to_dict(orient='records')
    if fit_dict_vect:
        X = dict_vector.fit_transform(dicts)
    else:
        X = dict_vector.transform(dicts)

    return X, dict_vector


In [31]:
def run_data_prep(dest_path: str, dataset: str = "green"):
    df_train = read_data_frame(green_taxi_01)
    df_val = read_data_frame(green_taxi_02)
    df_test = read_data_frame(green_taxi_03)

    target = 'duration'
    y_train = df_train[target].values
    y_val = df_val[target].values
    y_test = df_test[target].values

    dict_vector = DictVectorizer()
    X_train, dv = preprocess(df_train, dict_vector, fit_dict_vect=True)
    X_val, _ = preprocess(df_val, dict_vector, fit_dict_vect=False)
    X_test, _ = preprocess(df_test, dict_vector, fit_dict_vect=False)

    os.makedirs(dest_path, exist_ok=True)

    dump_pickle(dict_vector, os.path.join(dest_path, 'dv.pkl'))
    dump_pickle((X_train, y_train), os.path.join(dest_path, 'train.pkl'))
    dump_pickle((X_val, y_val), os.path.join(dest_path, 'val.pkl'))
    dump_pickle((X_test, y_test), os.path.join(dest_path, 'test.pkl'))


In [29]:
run_data_prep(dest_path="output")

In [30]:
# Q2
# - dv.pkl
# - test.pkl
# - train.pkl
# - val.pkl

In [36]:
import os
import pickle
import mlflow

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [37]:
def load_pickle(filename: str):
    with open(filename, 'rb') as f_in:
        return pickle.load(f_in)

In [38]:
def run_train(data_path: str):
    with mlflow.start_run():
        mlflow.sklearn.autolog()
        X_train, y_train = load_pickle(os.path.join(data_path, 'train.pkl'))
        X_val, y_val = load_pickle(os.path.join(data_path, 'val.pkl'))

        rf = RandomForestRegressor(max_depth=10, random_state=0)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)

        rmse = root_mean_squared_error(y_val, y_pred)
        return rmse

In [39]:
run_train('output')

2026/05/26 15:15:46 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


6.20744281479805

In [40]:
# Q3 - 2

In [45]:
import os
import pickle
import mlflow
import numpy as np
from hyperopt import STATUS_OK, Trials, fmin, hp, tpe
from hyperopt.pyll import scope
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("random-forest-hyperopt")

<Experiment: artifact_location='/Users/julia/Desktop/Projects/mlops-zoomcamp-exercises/02-experiment-tracking/artifacts/6', creation_time=1779778765139, experiment_id='6', last_update_time=1779778765139, lifecycle_stage='active', name='random-forest-hyperopt', tags={}, trace_location=None, workspace='default'>

In [42]:

def load_pickle(filename: str):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)

In [47]:
def run_optimization(data_path: str, num_trials: int):

    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))

    def objective(params):
        with mlflow.start_run():
            mlflow.log_params(params)

            rf = RandomForestRegressor(**params)
            rf.fit(X_train, y_train)
            y_pred = rf.predict(X_val)
            rmse = root_mean_squared_error(y_val, y_pred)
            mlflow.log_metric("rmse", rmse)

            return {'loss': rmse, 'status': STATUS_OK}

    search_space = {
        'max_depth': scope.int(hp.quniform('max_depth', 1, 20, 1)),
        'n_estimators': scope.int(hp.quniform('n_estimators', 10, 50, 1)),
        'min_samples_split': scope.int(hp.quniform('min_samples_split', 2, 10, 1)),
        'min_samples_leaf': scope.int(hp.quniform('min_samples_leaf', 1, 4, 1)),
        'random_state': 42
    }

    rstate = np.random.default_rng(42)  # for reproducible results
    fmin(
        fn=objective,
        space=search_space,
        algo=tpe.suggest,
        max_evals=num_trials,
        trials=Trials(),
        rstate=rstate
        )

In [48]:
run_optimization('output', num_trials=15)

  0%|          | 0/15 [00:00<?, ?trial/s, best loss=?]

2026/05/26 16:10:34 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run zealous-donkey-240 at: http://127.0.0.1:5000/#/experiments/6/runs/792e0b569caf4f9d956381d7a42c21e0

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6

  7%|▋         | 1/15 [00:10<02:20, 10.07s/trial, best loss: 6.156391570755012]

2026/05/26 16:10:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run vaunted-shrew-541 at: http://127.0.0.1:5000/#/experiments/6/runs/e4b5b317f3424fc5a5c4fb1ee63c7b0f

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6                   

 13%|█▎        | 2/15 [00:15<01:35,  7.33s/trial, best loss: 6.156391570755012]

...

🏃 View run nimble-auk-68 at: http://127.0.0.1:5000/#/experiments/6/runs/9ad18836e8994afc91a26f37e9e942ff

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6                    

 87%|████████▋ | 13/15 [01:25<00:11,  5.92s/trial, best loss: 6.132177499893072]

2026/05/26 16:11:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run auspicious-eel-627 at: http://127.0.0.1:5000/#/experiments/6/runs/683e9280ac8b4ac88f96d73358bd7bd0

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6                    

 93%|█████████▎| 14/15 [01:31<00:05,  5.87s/trial, best loss: 6.132177499893072]

2026/05/26 16:12:04 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



🏃 View run shivering-pug-37 at: http://127.0.0.1:5000/#/experiments/6/runs/3ba53760ba834ef98cd5ec64f02e2902

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/6                    

100%|██████████| 15/15 [01:38<00:00,  6.57s/trial, best loss: 6.132177499893072]


In [49]:

from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

In [50]:
HPO_EXPERIMENT_NAME = "random-forest-hyperopt"
EXPERIMENT_NAME = "random-forest-best-models"
RF_PARAMS = ['max_depth', 'n_estimators', 'min_samples_split', 'min_samples_leaf', 'random_state']

In [51]:
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.sklearn.autolog()

2026/05/26 16:36:16 INFO mlflow.tracking.fluent: Experiment with name 'random-forest-best-models' does not exist. Creating a new experiment.


In [52]:
def load_pickle(filename):
    with open(filename, "rb") as f_in:
        return pickle.load(f_in)

In [60]:
def train_and_log_model(data_path, params):
    X_train, y_train = load_pickle(os.path.join(data_path, "train.pkl"))
    X_val, y_val = load_pickle(os.path.join(data_path, "val.pkl"))
    X_test, y_test = load_pickle(os.path.join(data_path, "test.pkl"))

    with mlflow.start_run():
        new_params = {}
        for param in RF_PARAMS:
            new_params[param] = int(params[param])

        rf = RandomForestRegressor(**new_params)
        rf.fit(X_train, y_train)

        # Evaluate model on the validation and test sets
        val_rmse = root_mean_squared_error(y_val, rf.predict(X_val))
        mlflow.log_metric("val_rmse", val_rmse)
        test_rmse = root_mean_squared_error(y_test, rf.predict(X_test))
        mlflow.log_metric("test_rmse", test_rmse)

        mlflow.sklearn.log_model(
            sk_model=rf,
            name="model"
        )

In [64]:
def run_register_model(data_path: str, top_n: int):

    client = MlflowClient()

    # Retrieve the top_n model runs and log the models
    experiment = client.get_experiment_by_name(HPO_EXPERIMENT_NAME)
    runs = client.search_runs(
        experiment_ids=experiment.experiment_id,
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=top_n,
        order_by=["metrics.rmse ASC"]
    )
    for run in runs:
        train_and_log_model(data_path=data_path, params=run.data.params)

    # Select the model with the lowest test RMSE
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    best_run = client.search_runs(    
        experiment_ids=[experiment.experiment_id],
        filter_string='',
        run_view_type=ViewType.ACTIVE_ONLY,
        max_results=5,
        order_by=['metrics.test_rmse ASC']
    )
    run_id = best_run[0].info.run_id
    model_uri = f"runs:/{run_id}/model"

    # Register the best model
    mlflow.register_model(model_uri, name=EXPERIMENT_NAME)

In [65]:
run_register_model('output', 15)

2026/05/26 17:04:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/26 17:05:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run fortunate-rook-345 at: http://127.0.0.1:5000/#/experiments/7/runs/c0fc8831812b46d5bc174c7cb9473190
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


2026/05/26 17:05:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/26 17:05:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run treasured-shrike-865 at: http://127.0.0.1:5000/#/experiments/7/runs/69f84df6acfd42b4bd5aab4ca52fe4a4
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


2026/05/26 17:05:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/26 17:05:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

2026/05/26 17:07:05 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/26 17:07:10 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run thoughtful-panda-759 at: http://127.0.0.1:5000/#/experiments/7/runs/27261fefb2194cb3ba1c7842e04e8130
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


2026/05/26 17:07:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/26 17:07:18 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run chill-chimp-860 at: http://127.0.0.1:5000/#/experiments/7/runs/207abd91d2814f71be5746946c466581
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/7


Registered model 'random-forest-best-models' already exists. Creating a new version of this model...
2026/05/26 17:07:21 WARNING mlflow.tracking._model_registry.fluent: Run with id c0fc8831812b46d5bc174c7cb9473190 has no artifacts at artifact path 'model', registering model based on models:/m-6200cedd6d1f404d9f8e36c512b85368 instead
2026/05/26 17:07:22 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: random-forest-best-models, version 1
Created version '1' of model 'random-forest-best-models'.


In [66]:
# Q6
# 5.8